#### torch.nn.Module.state_dict():返回一个包含模块完整状态引用的字典。
参数：

    destination (dict, 可选) – 如果提供，模块的状态将被更新到 dict 中，并返回相同的对象。否则，将创建一个 OrderedDict 并返回。默认： None 。
    prefix (str, 可选) – 添加到参数和缓冲区名称前缀，以组成 state_dict 中的键。默认： '' 。
    keep_vars (bool, 可选) – 默认情况下，state dict 中返回的 Tensor 会从 autograd 中分离。如果设置为 True ，则不会执行分离。默认： False 。

返回：
    
    包含模块完整状态的字典，返回类型为dict

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 定义一个简单的海洋模型(输入3个特征 -> 输出1个特征)
class SimpleOceanModel(nn.Module):
    def __init__(self):
        super(SimpleOceanModel, self).__init__()
        # 使用两层全连接网络，比单层更容易演示梯度及其非线性
        self.fc1 = nn.Linear(3, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc3 = nn.Linear(4, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))  # 隐藏层使用ReLU激活函数
        x = F.relu(self.fc2(x))  # 隐藏层使用ReLU激活函数
        x = self.fc3(x)  # 输出层不使用激活函数
        return x

# 初始化全局模型
global_model = SimpleOceanModel()
# 返回一个包含模块完整状态引用的字典
global_state = global_model.state_dict()

In [16]:
global_state  # 查看全局模型的状态字典

OrderedDict([('fc1.weight',
              tensor([[-0.5320,  0.0913, -0.3934],
                      [-0.5204, -0.4309, -0.1491],
                      [ 0.5757, -0.2726, -0.5331],
                      [ 0.5653,  0.2307, -0.2061],
                      [ 0.3978, -0.0487,  0.4900],
                      [-0.3786, -0.1782,  0.2140],
                      [ 0.3878, -0.3693, -0.2988],
                      [ 0.5489,  0.5028,  0.0954]])),
             ('fc1.bias',
              tensor([-0.2406,  0.5370,  0.2366,  0.0559,  0.5019, -0.2179,  0.2808, -0.1434])),
             ('fc2.weight',
              tensor([[-0.2757, -0.1676,  0.1894,  0.2813, -0.1840, -0.1754, -0.1837, -0.3146],
                      [ 0.2731, -0.0577,  0.3535,  0.2033, -0.3012,  0.0119, -0.1305,  0.3294],
                      [-0.0480,  0.1596, -0.1311, -0.2218,  0.1969,  0.2880,  0.3106,  0.1203],
                      [-0.0884,  0.2725, -0.2012, -0.0983,  0.2397, -0.2853,  0.2901, -0.3381]])),
             ('fc2.bias

In [6]:
global_state.keys()  # 查看全局模型的状态字典的键

odict_keys(['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias'])

#### torch.nn.Module.parameters():返回模型中所有层的可训练参数（通常是权重和偏置）的迭代器。
在PyTorch中，torch.nn.Module类（所有神经网络模块的基类）有一个方法parameters()，它返回一个包含该模块所有可训练参数（通常是权重和偏置）的迭代器。

In [19]:
global_model.parameters()  # 返回模型中所有可训练参数的迭代器对象  Module.parameters

# 查看所有参数
for param in global_model.parameters():
    print(param)
    print(f"形状: {param.shape}, 需要梯度: {param.requires_grad}")
    print("-----------------")

Parameter containing:
tensor([[-0.5320,  0.0913, -0.3934],
        [-0.5204, -0.4309, -0.1491],
        [ 0.5757, -0.2726, -0.5331],
        [ 0.5653,  0.2307, -0.2061],
        [ 0.3978, -0.0487,  0.4900],
        [-0.3786, -0.1782,  0.2140],
        [ 0.3878, -0.3693, -0.2988],
        [ 0.5489,  0.5028,  0.0954]], requires_grad=True)
形状: torch.Size([8, 3]), 需要梯度: True
-----------------
Parameter containing:
tensor([-0.2406,  0.5370,  0.2366,  0.0559,  0.5019, -0.2179,  0.2808, -0.1434],
       requires_grad=True)
形状: torch.Size([8]), 需要梯度: True
-----------------
Parameter containing:
tensor([[-0.2757, -0.1676,  0.1894,  0.2813, -0.1840, -0.1754, -0.1837, -0.3146],
        [ 0.2731, -0.0577,  0.3535,  0.2033, -0.3012,  0.0119, -0.1305,  0.3294],
        [-0.0480,  0.1596, -0.1311, -0.2218,  0.1969,  0.2880,  0.3106,  0.1203],
        [-0.0884,  0.2725, -0.2012, -0.0983,  0.2397, -0.2853,  0.2901, -0.3381]],
       requires_grad=True)
形状: torch.Size([4, 8]), 需要梯度: True
---------------

In [ ]:
# 获取参数数量
total_params = sum(p.numel() for p in global_model.parameters())
print(f"总参数数量: {total_params}")  # （8*3 + 8）+（4*8 + 4）+（1*4 + 1） = 73

# 只获取需要梯度的参数
trainable_params = [p for p in global_model.parameters() if p.requires_grad]
print(f"需要梯度的参数层数: {len(trainable_params)}")  # fc1.weight、fc1.bias、fc2.weight、fc2.bias、fc3.weight、fc3.bias

# 冻结某些层
for param in global_model.fc1.parameters():
    # param.requires_grad = False  # 请替换执行看效果
    param.requires_grad = True

总参数数量: 73
需要梯度的参数数量: 6


In [22]:
# named_parameters() - 带名称的参数
for name, param in global_model.named_parameters():
    print(f"{name}: {param}")
    print(f"形状: {param.shape}, 需要梯度: {param.requires_grad}")
    print("-----------------")

fc1.weight: Parameter containing:
tensor([[-0.5320,  0.0913, -0.3934],
        [-0.5204, -0.4309, -0.1491],
        [ 0.5757, -0.2726, -0.5331],
        [ 0.5653,  0.2307, -0.2061],
        [ 0.3978, -0.0487,  0.4900],
        [-0.3786, -0.1782,  0.2140],
        [ 0.3878, -0.3693, -0.2988],
        [ 0.5489,  0.5028,  0.0954]], requires_grad=True)
形状: torch.Size([8, 3]), 需要梯度: True
-----------------
fc1.bias: Parameter containing:
tensor([-0.2406,  0.5370,  0.2366,  0.0559,  0.5019, -0.2179,  0.2808, -0.1434],
       requires_grad=True)
形状: torch.Size([8]), 需要梯度: True
-----------------
fc2.weight: Parameter containing:
tensor([[-0.2757, -0.1676,  0.1894,  0.2813, -0.1840, -0.1754, -0.1837, -0.3146],
        [ 0.2731, -0.0577,  0.3535,  0.2033, -0.3012,  0.0119, -0.1305,  0.3294],
        [-0.0480,  0.1596, -0.1311, -0.2218,  0.1969,  0.2880,  0.3106,  0.1203],
        [-0.0884,  0.2725, -0.2012, -0.0983,  0.2397, -0.2853,  0.2901, -0.3381]],
       requires_grad=True)
形状: torch.Size([

model.parameters() vs model.state_dict() 深度解析

    特性	model.parameters()	      model.state_dict()
    返回类型	生成器/迭代器	            有序字典(OrderedDict)
    内容	只有可训练参数(Parameter对象)	    所有参数+缓冲区+模型结构信息
    使用场景	训练时传递给优化器	            模型保存/加载、推理部署
    包含梯度	包含.grad属性	            不包含梯度信息
    内存引用	直接引用参数对象	            参数的副本/快照

#### copy.deepcopy(global_model)  # 创建对象的完全独立副本，包括所有嵌套对象。
代码的作用：

    创建 global_model 的一个完整独立副本
    新副本 victim_model 与原始模型完全独立
    修改 victim_model 不会影响 global_model


copy.copy():浅拷贝，只拷贝最外层。适用于简单对象，没有嵌套结构

copy.deepcopy():深拷贝，递归拷贝所有嵌套对象。适用于复杂对象，如神经网络模型

In [5]:
import torch
import copy

# 原始模型
original_model = torch.nn.Linear(10, 1)

# 浅拷贝
shallow_copy = copy.copy(original_model)
# 深拷贝  
deep_copy = copy.deepcopy(original_model)

# 修改浅拷贝的参数
shallow_copy.weight.data[0, 0] = 999.0
deep_copy.weight.data[0, 0] = 888.0

print("原始模型参数:", original_model.weight.data[0, 0])  # 也会变成999！
print("浅拷贝参数:", shallow_copy.weight.data[0, 0])      # 999
print("深拷贝参数:", deep_copy.weight.data[0, 0])         # 888

原始模型参数: tensor(999.)
浅拷贝参数: tensor(999.)
深拷贝参数: tensor(888.)


In [ ]:
# 对于PyTorch模型，你也可以使用以下替代方案：
# 方法1：深拷贝（推荐）
victim_model = copy.deepcopy(global_model)

# 方法2：序列化再反序列化
victim_model = torch.load(torch.save(global_model))

# 方法3：手动复制状态字典
victim_model = SimpleOceanModel()
victim_model.load_state_dict(global_model.state_dict())

#### torch.autograd.grad(outputs, inputs):计算并返回 outputs 关于 inputs 的梯度之和。
是PyTorch自动求导机制中的一个函数，用于计算并返回输出相对于输入的梯度。
通常我们会在模型测试时通过torch.no_grad()来禁用梯度计算。

参数：

    outputs：dy，需要求导的张量（通常是损失函数，默认是标量）
    inputs：dx，需要计算梯度的张量（通常是模型参数）
    grad_outputs：相对于outputs的梯度，默认为1（即标量输出）
    retain_graph：如果为False，计算梯度后释放计算图；如果为True，则保留计算图，以便再次求导
    create_graph：如果为True，则构建导数的计算图，用于高阶求导
    only_inputs：如果为True，则只返回inputs的梯度；如果为False，则返回所有叶子节点的梯度
    allow_unused：如果为False，则当某些inputs未在计算图中使用时报错；如果为True，则返回None作为未使用的输入的梯度

返回类型：

    一个元组，包含与 inputs 长度相同的张量，每个张量对应 inputs 中对应元素的梯度。（梯度=求导）

In [77]:
import torch

# 简单示例
x = torch.tensor([2.0], requires_grad=True)  # requires_grad：是否需要计算梯度
y = x ** 2 + 3 * x + 1

# 计算 dy/dx
gradient = torch.autograd.grad(y, x)
print(f"dy/dx = {gradient[0]}")  # 输出: tensor([7.]) 因为 2*2 + 3 = 7

dy/dx = tensor([7.])


#### torch.autograd.grad()与loss.backward() 的区别
两者都是求导计算梯度，但是loss.backward()只能用于计算标量损失函数的梯度，而torch.autograd.grad()可以用于计算任意张量的梯度。

    方法	                返回值	      梯度存储位置	    使用场景            计算图                  内存效率
    loss.backward()	        None	      参数的 .grad 属性	    常规训练            默认会释放计算图          更高（释放图）
    torch.autograd.grad()	梯度元组	      直接返回梯度	    需要操作梯度时       默认保留计算图            更低（保留图）

训练流程：清零梯度 -> 前向传播 -> 计算损失 -> 反向传播 -> 更新参数。

    loss.backward()：用于常规训练，梯度累积在模型参数model.parameters()的.grad属性。
    torch.autograd.grad()：用于需要直接获取梯度值的情况，不累积梯度，更灵活。

    # 方法1: loss.backward()
    loss.backward()
    gradients_backward = [p.grad for p in model.parameters()]
    
    #方法2: torch.autograd.grad()
    gradients_direct = torch.autograd.grad(loss, model.parameters())
    # 两种方法得到的梯度是一样的

loss.backward()

    功能：计算损失对于所有需要梯度（requires_grad=True）的叶子节点的梯度，并将梯度累积到这些叶子节点的.grad属性中。
    返回值：无。
    使用场景：通常用于整个模型的训练，因为我们只需要调用一次就可以计算所有参数的梯度，然后使用优化器更新。

torch.autograd.grad()

    功能：计算并返回一组输出相对于一组输入的梯度。不会累积梯度到.grad属性，而是直接返回梯度的元组。
    返回值：梯度的元组，每个元素对应一个输入张量的梯度。
    使用场景：当需要直接获取梯度值而不想改变模型参数的.grad属性时，或者需要计算高阶导数（设置create_graph=True）时。

关键区别

    梯度存储位置：
        loss.backward()：将梯度存储在叶子节点的.grad属性中。
        torch.autograd.grad()：直接返回梯度，不存储到.grad属性。

    梯度累积：
        loss.backward()：默认会累积梯度（即每次调用会将梯度加到.grad上），所以通常我们在训练循环开始时用optimizer.zero_grad()清零。
        torch.autograd.grad()：只是计算并返回梯度，不会累积。

    使用灵活性：
        loss.backward()：适用于大多数训练场景，简单方便。
        torch.autograd.grad()：更灵活，可以计算非叶子节点的梯度，或者需要梯度值进行其他操作时。

In [100]:
import torch
import torch.nn as nn

# 创建模型和数据
model = nn.Linear(3, 1)
x = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=False)  # requires_grad=False将不可通过x.grad获取梯度
y_true = torch.tensor([[5.0]])

# 方法1: 使用 backward()
model.zero_grad()  # 清除之前的梯度
y_pred = model(x)
loss1 = nn.MSELoss()(y_pred, y_true)
loss1.backward()  # 反向传播，计算梯度
grads1 = [p.grad.clone() for p in model.parameters()]

# 方法2: 使用 autograd.grad()
y_pred = model(x)
loss2 = nn.MSELoss()(y_pred, y_true)
grads2 = torch.autograd.grad(loss2, model.parameters())  # 计算损失关于模型参数的梯度

print("两种方法梯度是否一致:", torch.allclose(grads1[0], grads2[0]))

None
两种方法梯度是否一致: True


#### 示例对比

In [26]:
# 假设我们有一个简单的函数：y = x^2，我们想计算在x=2处的梯度。

# 使用backward():
x = torch.tensor(2.0, requires_grad=True)  # 注意：这里x需要是可微的，即requires_grad=True，可通过x.grad获取梯度
y = x ** 2

y.backward()
print(x.grad)  # 输出: tensor(4.)

tensor(4.)


In [27]:
# 使用autograd.grad():
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2

grad = torch.autograd.grad(y, x)  # dy / dx
print(grad[0])  # 输出: tensor(4.)

tensor(4.)


在联邦学习攻击代码中的使用
在梯度反演攻击中，我们使用torch.autograd.grad()是因为：

    我们不需要更新模型参数，而是需要直接拿到梯度值用于攻击。
    我们可能需要对梯度进行进一步的操作（比如计算梯度匹配损失）。
    我们不想干扰模型参数的.grad属性，因为攻击过程中我们可能还需要正常训练（伪装）。

在攻击代码中，我们看到：

    # 计算虚拟数据的梯度
    dummy_gradients = torch.autograd.grad(
        dummy_loss, 
        self.model.parameters(), 
        create_graph=True  # 因为我们需要对虚拟数据求导，所以需要创建计算图
    )
这里使用torch.autograd.grad()是因为我们需要虚拟数据对应的梯度，并且我们还需要对这个梯度进行反向传播（所以需要create_graph=True），而不是为了更新模型。


#### torch.nn.Module 常用方法和属性

In [82]:
# 下面归纳几种查看对象的属性与方法的方式
model = SimpleOceanModel()

print(dir(model))  # 打印所有属性（方法也算属性，方法是可调用的属性）
print(model.__dict__)  # 打印构造函数中的属性


# 获取所有属性和方法
all_attrs = dir(model)
# 过滤出方法  判断每个属性是否为可调用的（即方法）
methods = [attr for attr in all_attrs if callable(getattr(model, attr))]
print(methods)



['T_destination', '__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_apply', '_backward_hooks', '_backward_pre_hooks', '_buffers', '_call_impl', '_compiled_call_impl', '_forward_hooks', '_forward_hooks_always_called', '_forward_hooks_with_kwargs', '_forward_pre_hooks', '_forward_pre_hooks_with_kwargs', '_get_backward_hooks', '_get_backward_pre_hooks', '_get_name', '_is_full_backward_hook', '_load_from_state_dict', '_load_state_dict_post_hooks', '_load_state_dict_pre_hooks', '_maybe_warn_non_full_backward_hook', '_modules', '_named_members', '_non_persistent_buffers_set', '_parameters', '_register_load_state_dict_pre_hoo

常用属性

In [94]:
model = SimpleOceanModel()

# 1. training - 模型模式
print(f"训练模式: {model.training}")

# 2. _modules - 子模块字典
print(f"子模块: {model._modules}")

# 3. _parameters - 参数字典
print(f"参数: {list(model._parameters.keys())}")

训练模式: True
子模块: OrderedDict([('fc1', Linear(in_features=3, out_features=8, bias=True)), ('fc2', Linear(in_features=8, out_features=4, bias=True)), ('fc3', Linear(in_features=4, out_features=1, bias=True))])
参数: []


常用方法

In [96]:
# parameters() - 所有参数
for param in model.parameters():
    print(param.shape)

torch.Size([8, 3])
torch.Size([8])
torch.Size([4, 8])
torch.Size([4])
torch.Size([1, 4])
torch.Size([1])


In [12]:
# named_parameters() - 带名称的参数
for name, param in model.named_parameters():
    print(f"{name}: {param.shape}")

fc1.weight: torch.Size([8, 3])
fc1.bias: torch.Size([8])
fc2.weight: torch.Size([4, 8])
fc2.bias: torch.Size([4])
fc3.weight: torch.Size([1, 4])
fc3.bias: torch.Size([1])


In [13]:
# state_dict() - 模型状态字典
state_dict = model.state_dict()
print("状态字典键:", state_dict.keys())

状态字典键: odict_keys(['fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias', 'fc3.weight', 'fc3.bias'])


In [14]:
# load_state_dict() - 加载状态字典（也就是加载模型的参数：权重+偏置）
model.load_state_dict(state_dict)

<All keys matched successfully>

In [15]:
# 训练模式
model.train()
print(f"训练模式: {model.training}")  # True

训练模式: True


In [16]:
# 评估模式
model.eval()
print(f"训练模式: {model.training}")  # False

训练模式: False


In [20]:
# 移动到GPU
if torch.cuda.is_available():
    model.cuda()

In [18]:
# 移动到CPU
model.cpu()

SimpleOceanModel(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=1, bias=True)
)

In [19]:
# 使用to方法
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SimpleOceanModel(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=1, bias=True)
)

In [21]:
# 应用函数到所有子模块
def init_weights(m):
    if isinstance(m, nn.Linear):  # 初始化线性层的权重
        torch.nn.init.xavier_uniform_(m.weight)  # 初始化权重
        m.bias.data.fill_(0.01)  # 初始化偏置项

model.apply(init_weights)

SimpleOceanModel(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=1, bias=True)
)

In [22]:
# 获取所有子模块
for module in model.modules():
    print(module)

SimpleOceanModel(
  (fc1): Linear(in_features=3, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=4, bias=True)
  (fc3): Linear(in_features=4, out_features=1, bias=True)
)
Linear(in_features=3, out_features=8, bias=True)
Linear(in_features=8, out_features=4, bias=True)
Linear(in_features=4, out_features=1, bias=True)


In [23]:
# 获取直接子模块
for child in model.children():
    print(child)

Linear(in_features=3, out_features=8, bias=True)
Linear(in_features=8, out_features=4, bias=True)
Linear(in_features=4, out_features=1, bias=True)


#### 深入对比 torch.autograd.grad() 与 loss.backward() 的区别

PyTorch模型训练完整流程(请单独测试)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# 1. 定义模型、优化器、损失函数
class SimpleOceanModel(nn.Module):
    def __init__(self):
        super(SimpleOceanModel, self).__init__()
        # 使用两层全连接网络，比单层更容易演示梯度及其非线性
        self.fc1 = nn.Linear(3, 8)
        self.fc2 = nn.Linear(8, 4)
        self.fc3 = nn.Linear(4, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))  # 隐藏层使用ReLU激活函数
        x = F.relu(self.fc2(x))  # 隐藏层使用ReLU激活函数
        x = self.fc3(x)  # 输出层不使用激活函数
        return x

model = SimpleOceanModel()
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# 模拟一批数据
inputs = torch.randn(16, 3)  # batch_size=16, features=3
targets = torch.randn(16, 1)

# 训练循环
for epoch in range(3):
    
    # 2. 梯度清零 - 关键步骤！
    optimizer.zero_grad()
    
    # 3. 前向传播
    outputs = model(inputs)
    
    # 4. 计算损失
    loss = criterion(outputs, targets)
    
    # 5. 反向传播
    loss.backward()
    
    # 6. 更新权重
    optimizer.step()
    
    if epoch % 1 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 1.3876
Epoch 1, Loss: 1.3800
Epoch 2, Loss: 1.3726


步骤1：梯度清零 (optimizer.zero_grad())

In [16]:
# 为什么需要梯度清零？
# 因为PyTorch会累积梯度，如果不清零，梯度会不断累加

# 演示梯度累积问题
x = torch.tensor([2.0], requires_grad=True)  # x可微，即requires_grad=True，可通过x.grad获取梯度
y = x ** 2
optimizer = optim.SGD([x], lr=0.1)  # 这里用[x]模拟了模型的参数model.parameters()的效果

# 第一次反向传播
y.backward()
print(f"第一次梯度: {x.grad}")  # tensor([4.])
# 不清零，直接第二次反向传播
y.backward()  # 注意：这实际上会报错，因为图被释放了，这里只是概念说明
# 如果允许，梯度会变成: tensor([8.]) = 4 + 4

第一次梯度: tensor([4.])


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

计算图与梯度流的深入理解

In [ ]:
import torch

def computation_graph_demo():
    """理解PyTorch的计算图和梯度流"""
    
    # 创建计算图
    x = torch.tensor(2.0, requires_grad=True)  # 输入
    w = torch.tensor(3.0, requires_grad=True)  # 权重
    b = torch.tensor(1.0, requires_grad=True)  # 偏置
    
    # 前向传播
    y = w * x + b      # y = 3*2 + 1 = 7
    z = y ** 2         # z = 7^2 = 49
    
    print("计算图:")
    print(f"x: {x}, w: {w}, b: {b}")
    print(f"y = w*x + b = {y}")
    print(f"z = y^2 = {z}")
    
    # 方法1: backward() - 自动填充 .grad
    z.backward()
    print("\n使用 backward() 后的梯度:")
    print(f"∂z/∂x = {x.grad}")  # ∂z/∂x = ∂z/∂y * ∂y/∂x = 2y * w = 2*7*3 = 42
    print(f"∂z/∂w = {w.grad}")  # ∂z/∂w = ∂z/∂y * ∂y/∂w = 2y * x = 2*7*2 = 28
    print(f"∂z/∂b = {b.grad}")  # ∂z/∂b = ∂z/∂y * ∂y/∂b = 2y * 1 = 2*7 = 14
    
    # 重置梯度
    x.grad = None
    w.grad = None
    b.grad = None
    
    # # 方法2: autograd.grad() - 手动获取梯度
    # gradients = torch.autograd.grad(z, [x, w, b])
    # print("\n使用 autograd.grad() 获取的梯度:")
    # print(f"∂z/∂x = {gradients[0]}")
    # print(f"∂z/∂w = {gradients[1]}")
    # print(f"∂z/∂b = {gradients[2]}")

computation_graph_demo()

计算图:
x: 2.0, w: 3.0, b: 1.0
y = w*x + b = 7.0
z = y^2 = 49.0

使用 autograd.grad() 获取的梯度:
∂z/∂x = 42.0
∂z/∂w = 28.0
∂z/∂b = 14.0


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

上面代码报错原因分析：

    这个错误是因为在第一次调用z.backward()时，PyTorch默认会释放计算图以节省内存。因此，当我们尝试第二次求导时报错了（无论是通过backward()还是autograd.grad()）时，计算图已经不存在了。

    解决方案有两种：
    1.在第一次反向传播时设置retain_graph=True、然后在第二次反向传播之后释放。
    （注意：当我们使用retain_graph=True时，计算图会被保留，所以我们可以进行第二次反向传播。但是，当我们不再需要计算图时，应该释放它，否则会占用内存。在这个例子中，函数执行完毕后，这些局部变量会被销毁，所以不用担心。）

    2.我们不用backward()，而是两次都用autograd.grad()，或者第一次用autograd.grad()，第二次用backward()，只要保留计算图。

In [3]:
import torch

def computation_graph_demo():
    """理解PyTorch的计算图和梯度流"""
    
    # 创建计算图
    x = torch.tensor(2.0, requires_grad=True)  # 输入
    w = torch.tensor(3.0, requires_grad=True)  # 权重
    b = torch.tensor(1.0, requires_grad=True)  # 偏置
    
    # 前向传播
    y = w * x + b      # y = 3*2 + 1 = 7
    z = y ** 2         # z = 7^2 = 49
    
    print("计算图:")
    print(f"x: {x}, w: {w}, b: {b}")
    print(f"y = w*x + b = {y}")
    print(f"z = y^2 = {z}")
    
    # 方法1: backward() - 自动填充 .grad
    z.backward(retain_graph=True)  # 保留计算图
    print("\n使用 backward() 后的梯度:")
    print(f"∂z/∂x = {x.grad}")  # ∂z/∂x = ∂z/∂y * ∂y/∂x = 2y * w = 2*7*3 = 42
    print(f"∂z/∂w = {w.grad}")  # ∂z/∂w = ∂z/∂y * ∂y/∂w = 2y * x = 2*7*2 = 28
    print(f"∂z/∂b = {b.grad}")  # ∂z/∂b = ∂z/∂y * ∂y/∂b = 2y * 1 = 2*7 = 14
    
    # 重置梯度
    x.grad = None
    w.grad = None
    b.grad = None

    # 注意：我们已经通过backward()得到了梯度并存储在x.grad等中，然后我们重置了梯度为None，
    # 但是计算图还保留着，所以第二次用autograd.grad()是可以的。
    
    # 方法2: autograd.grad() - 手动获取梯度
    gradients = torch.autograd.grad(z, [x, w, b])
    print("\n使用 autograd.grad() 获取的梯度:")
    print(f"∂z/∂x = {gradients[0]}")
    print(f"∂z/∂w = {gradients[1]}")
    print(f"∂z/∂b = {gradients[2]}")

computation_graph_demo()

计算图:
x: 2.0, w: 3.0, b: 1.0
y = w*x + b = 7.0
z = y^2 = 49.0

使用 backward() 后的梯度:
∂z/∂x = 42.0
∂z/∂w = 28.0
∂z/∂b = 14.0

使用 autograd.grad() 获取的梯度:
∂z/∂x = 42.0
∂z/∂w = 28.0
∂z/∂b = 14.0


详细代码对比

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

def compare_gradient_methods():
    # 创建相同的模型和数据
    model1 = nn.Linear(3, 1)
    model2 = nn.Linear(3, 1)
    
    # 确保两个模型初始状态相同
    model2.load_state_dict(model1.state_dict())  # 加载model1的参数到model2
    
    x = torch.tensor([[1.0, 2.0, 3.0]])
    y_true = torch.tensor([[2.0]])
    criterion = nn.MSELoss()  # 均方误差
    optimizer = optim.Adam(model1.parameters())
    
    print("=== 方法1: loss.backward() ===")
    # 方法1: 使用 backward()
    y_pred1 = model1(x)
    loss1 = criterion(y_pred1, y_true)
    loss1.backward()  # 梯度计算并存储
    # 这里没有进行模型梯度更新（模型参数未改变），正常应该会执行optimizer.step()的
    
    print("梯度存储在参数的.grad属性:")
    for name, param in model1.named_parameters():
        print(f"{name}: {param.grad}")

    optimizer.step()
    
    print("\n=== 方法2: torch.autograd.grad() ===")
    # 方法2: 使用 autograd.grad()
    y_pred2 = model2(x)
    loss2 = criterion(y_pred2, y_true)
    
    # 手动计算梯度
    gradients = torch.autograd.grad(
        outputs=loss2,
        inputs=model2.parameters(),
        create_graph=False,  # 设为True可以计算高阶导数
        retain_graph=True    # 保留计算图以便后续使用
    )
    
    print("梯度作为元组返回:")
    for i, grad in enumerate(gradients):
        print(f"参数 {i}: {grad}")
    
    # 验证两种方法结果是否相同
    manual_grads = [p.grad for p in model1.parameters()]
    auto_grads = gradients
    
    print("\n=== 验证一致性 ===")
    for i, (mg, ag) in enumerate(zip(manual_grads, auto_grads)):
        print(f"参数 {i} 梯度一致: {torch.allclose(mg, ag)}")

compare_gradient_methods()

=== 方法1: loss.backward() ===
梯度存储在参数的.grad属性:
weight: tensor([[-2.1051, -4.2103, -6.3154]])
bias: tensor([-2.1051])

=== 方法2: torch.autograd.grad() ===
梯度作为元组返回:
参数 0: tensor([[-2.1051, -4.2103, -6.3154]])
参数 1: tensor([-2.1051])

=== 验证一致性 ===
参数 0 梯度一致: True
参数 1 梯度一致: True


分析这行代码：for i, (mg, ag) in enumerate(zip(manual_grads, auto_grads))：

In [ ]:
manual_grads = [grad1, grad2, grad3]  # 来自方法1的梯度
auto_grads = [gradA, gradB, gradC]    # 来自方法2的梯度

# 步骤1：zip(manual_grads, auto_grads)
# 生成：[(grad1, gradA), (grad2, gradB), (grad3, gradC)]

# 步骤2：enumerate(zip(...))
# 生成：[(0, (grad1, gradA)), (1, (grad2, gradB)), (2, (grad3, gradC))]

# 步骤3：i, (mg, ag) 解包
# 第一次循环：i=0, mg=grad1, ag=gradA
# 第二次循环：i=1, mg=grad2, ag=gradB  
# 第三次循环：i=2, mg=grad3, ag=gradC

In [30]:
names = ['Alice', 'Bob', 'Charlie']
scores = [85, 92, 78]

# 传统写法
for i in range(len(names)):
    print(f"{i}: {names[i]} - {scores[i]}")
print('-------------------')

# 使用zip和enumerate的优雅写法
for i, (name, score) in enumerate(zip(names, scores)):
    print(f"{i}: {name} - {score}")

0: Alice - 85
1: Bob - 92
2: Charlie - 78
-------------------
0: Alice - 85
1: Bob - 92
2: Charlie - 78


类似的常用模式

In [32]:
# 同时遍历多个列表
list1 = [1, 2, 3]
list2 = ['a', 'b', 'c']
for x, y in zip(list1, list2):
    print(x, y)
print('-'*15)

# 遍历字典的键值对
data = {'a': 1, 'b': 2, 'c': 3}
for key, value in data.items():
    print(key, value)
print('-'*15)

# 需要索引的遍历
items = ['apple', 'banana', 'orange']
for idx, item in enumerate(items):
    print(f"{idx}: {item}")

1 a
2 b
3 c
---------------
a 1
b 2
c 3
---------------
0: apple
1: banana
2: orange


实际应用场景示例

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

def practical_scenarios():
    """不同场景下的梯度计算选择"""
    
    # 场景1: 常规训练 - 使用 backward()
    def normal_training():
        model = nn.Linear(5, 1)
        optimizer = optim.Adam(model.parameters())
        
        for epoch in range(2):
            optimizer.zero_grad()
            output = model(torch.randn(10, 5))
            loss = nn.MSELoss()(output, torch.randn(10, 1))
            loss.backward()  # 简单高效
            for name, param in model.named_parameters():
                print(f"第{epoch}次前向传播, {name} 的梯度:")
                print(f"{param.grad}")
                
            print("模型参数更新前:", model.state_dict())
            optimizer.step()
            print("模型参数更新后:", model.state_dict())
    
    # 场景2: 梯度分析 - 使用 autograd.grad()
    def gradient_analysis():
        model = nn.Sequential(
            nn.Linear(5, 10),
            nn.ReLU(),
            nn.Linear(10, 1)
        )
        
        x = torch.randn(1, 5, requires_grad=True)
        output = model(x)
        loss = output.sum()
        
        # 分析每一层的梯度
        gradients = {}
        for name, param in model.named_parameters():
            grad = torch.autograd.grad(loss, param, retain_graph=True)[0]
            gradients[name] = grad
            print(f"{name} 的梯度范数: {grad.norm().item():.4f}")
    
    # 场景3: 梯度攻击（如你的联邦学习代码） - 使用 autograd.grad()
    def gradient_attack_simulation():
        model = nn.Linear(3, 1)
        sensitive_data = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
        
        # 受害者计算梯度（模拟上传）
        output = model(sensitive_data)
        target_loss = output.sum()
        target_gradients = torch.autograd.grad(target_loss, model.parameters())
        
        print("目标梯度:")
        for grad in target_gradients:
            print(grad)
        
        # 攻击者尝试恢复数据（使用这些梯度）
        return target_gradients
    
    print("=== 场景1: 常规训练 ===")
    normal_training()
    
    print("\n=== 场景2: 梯度分析 ===")
    gradient_analysis()
    
    print("\n=== 场景3: 梯度攻击模拟 ===")
    gradient_attack_simulation()

practical_scenarios()

=== 场景1: 常规训练 ===
第0次前向传播, weight 的梯度:
tensor([[ 0.9575, -0.2014,  0.0541, -0.0255, -1.4552]])
第0次前向传播, bias 的梯度:
tensor([-0.4574])
模型参数更新前: OrderedDict([('weight', tensor([[ 0.1411, -0.2088, -0.3329, -0.3153, -0.3837]])), ('bias', tensor([-0.1771]))])
模型参数更新后: OrderedDict([('weight', tensor([[ 0.1401, -0.2078, -0.3339, -0.3143, -0.3827]])), ('bias', tensor([-0.1761]))])
第1次前向传播, weight 的梯度:
tensor([[ 0.3880, -1.0948, -2.0491, -0.6120, -2.5747]])
第1次前向传播, bias 的梯度:
tensor([0.3778])
模型参数更新前: OrderedDict([('weight', tensor([[ 0.1401, -0.2078, -0.3339, -0.3143, -0.3827]])), ('bias', tensor([-0.1761]))])
模型参数更新后: OrderedDict([('weight', tensor([[ 0.1392, -0.2070, -0.3331, -0.3135, -0.3817]])), ('bias', tensor([-0.1761]))])

=== 场景2: 梯度分析 ===
0.weight 的梯度范数: 1.0623
0.bias 的梯度范数: 0.5739
2.weight 的梯度范数: 1.5674
2.bias 的梯度范数: 1.0000

=== 场景3: 梯度攻击模拟 ===
目标梯度:
tensor([[1., 2., 3.]])
tensor([1.])
